# Image Segmentation
(3 esercizi)

## 1: Pipeline di training di una U-Net sul dataset PennFudanPed

- Caricamento e preparazione del dataset (immagini: 256x256, in tensori e normalizzate; maschere: 256x256, in tensori binari)
- Creazione del dataset personalizzato
- Split in training e validation (80 - 20)
- Definizione della U-Net semplificata (3 encoder, bottleneck, 3 decoder con connection, classificatore finale a 2 classi)
- Impostazione training (CrossEntrpy, Adam, CUDA)
- Addestramento, Validazione, Salvataggio del modello (esporto i pesi)

Esercizio: Utilizzare le loss Focal Loss + Dice Loss di Hugging Face

In [ ]:
from pathlib import Path
from typing import Tuple
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

from transformers.model.detr.modeling_detr import sigmoid_focal_loss, dice_loss

ROOT_DIR = Path(__file__).resolve().parents[1]
DATA_ROOT = ROOT_DIR / 'data' / / 'PennFudanPed'
IMAGE_SIZE = 256
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
VAL_SPLIT = 0.2
torch.manual_seed(0)

In [ ]:
class PennFudanSegmentation(Dataset):
  '''Dataset semplice: restituisce immagini RGB e maschere binarie (pedone vs sfondo)'''

  def __init__(self, root: Path = DATA_ROOT) -> None:
    self.root = Path(root)
    self.image_dir = self.root / 'PNGImages'
    self.mask_dir = self.root / 'PedMasks'
    if not self.image_dir.exists() or not self.mask_dir.exists():
      raise FileNotFoundError(f'Struttura PennFudanPed non trovata in {self.root}. Assicurati di avere data/PennFudanPed con PNGImages e PedMasks.')

    self.images = sorted(self.image_dir.glob('.png'))
    if not self.images:
      raise RuntimeError(f'Nessuna immagine trovata in {self.image_dir}')

    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    self.images_transform = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    self.mask_transform = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.NEAREST),
        T.PILToTensor(),
        T.Lambda(lambda t: (t > 0).long().sqeeze(0)),
    ])

  def __len__(self) -> int:
    return len(self.images)

  def _getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
    image_path = self.images[idx]
    _mask_name = image_path.name.split('.')
    mask_name = f'{_mask_name[0]}_mask.{_mask_name[1]}'
    mask_path = self.mask_dir / mask_name

    image = Image.open(image_path).convert('RGB')
    mask = Image.open(mask_path)

    image = self.image_transform(image)
    mask = self.mask_transform(mask)
    return image, mask

In [ ]:
class UNet(nn.Module):
  def __init__(self, num_classes: int = 2) -> None:
    super().__init__()
    self.enc1 = self.conv_block(3, 32)
    self.enc2 = self.conv_block(32, 64)
    self.enc3 = self.conv_block(64, 128)
    self.pool = nn.MaxPool2d(2)

    self.bottleneck = self.conv_block(128, 256)

    self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
    self.dec3 = self.conv_block(256, 128)
    self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
    self.dec2 = self.conv_block(128, 64)
    self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
    self.dec1 = self.conv_block(64, 32)
    self.classifier = nn.Conv2d(32, num_classes, 1)

  def conv_block(self, in_channels: int, out_channels: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.ReLU(inplace=True),
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x1 = self.enc1(x)
    x2 = self.enc2(self.pool(x1))
    x3 = self.enc3(self.pool(x2))
    x4 = self.bottleneck(self.pool(x3))

    x = self.up3(x4)
    x = self.dec3(torch.cat([x, x3], dim=1))
    x = self.up2(x)
    x = self.dec2(torch.cat([x, x2], dim=1))
    x = self.up1(x)
    x = self.dec1(torch.cat([x, x1], dim=1))
    return self.classifier(x)

In [ ]:
dataset = PennFudanSegmentation(DATA_ROOT)
val_len = max(1, int(len(dataset) * VAL_SPLIT))
train_len = len(dataset) - val_len
train_set, val_set = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(num_classes=2).to(device)

# Loss iniziale
# criterion = nn.CrossEntropyLoss()

# Loss dell'esercizio
def criterion(logits, masks):
  masks = masks.unsqueeze(1).float()

  focal = sigmoid_focal_loss(logits, masks, reduction='mean')
  dice = dice_loss(logits, masks, reduction='mean')
  return focal + dice

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- Training ---
model.train()
train_loss = 0.0
for images, masks in train_loader:
  images, masks = images.to(device), masks.to(device)
  optimizer.zero_grad()
  logits = model(images)
  loss = criterion(logits, masks)

  loss.backward()
  optimizer.step()
  train_loss += loss.item() * images.size(0)
train_loss /= len(train_loader.dataset)
print(f'Loss training: {train_loss:.4f}')

# --- Validation ---
model.eval()
val_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
  for images, masks in val_loader:
    images, masks = images.to(device), masks.to(device)
    logits = model(images)
    loss = criterion(logits, masks)
    val_loss += loss.item() * images.size(0)

    preds = logits.argmax(dim=1)
    correct += (preds == masks).sum().item()
    total += masks.numel()

val_loss /= len(val_loader.dataset)
pixel_acc = correct / total if total else 0.0
print(f'Loss validation: {val_loss:.4f}, Pixel accuracy: {pixel_acc:.3f}')

torch.save(model.state_dict(), 'tiny_unet_pennfudan.pth')
print('Pesi salvati in tiny_unet_pennfudan.pth')

## 2: Pipeline di training di SegFormer sul dataset PennFudanPed
- Configurazioni iniziali (percorso del dataset, dim immagini, checkpoint Hugging Face)
- Creazione del dataset personalizzato
- Costruzione del modello SegFormer (SegformerForSemanticSegmentation)
- Upsampling dei logits
- Logging degli hidden states
- Preparazione del training (divisione del dataset, DataLoader, initialization)
- Addestramento, Validazione, Salvataggio del modello (esporto i pesi)

In [ ]:
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

ROOT_DIR = Path(__file__).resolve().parents[1]
DATA_ROOT = ROOT_DIR / "data" / "PennFudanPed"
IMAGE_SIZE = 256
BATCH_SIZE = 4
LEARNING_RATE = 5e-4
VAL_SPLIT = 0.2
SEGFORMER_CHECKPOINT = "nvidia/segformer-b0-finetuned-ade-512-512"
# SEGFORMER_CHECKPOINT = "nvidia/mit-b0"
torch.manual_seed(0)

In [ ]:
def build_segformer(num_classes: int) -> SegformerForSemanticSegmentation:
  model = SegformerForSemanticSegmentation.from_pretrained(
      SEGFORMER_CHECKPOINT,
      num_labels = num_classes,
      ignore_mismatched_sizes = True
  )
  model.config.output_hidden_state = True
  return model

def log_hidden_states(hidden_states, tag: str) -> None:
  if not hidden_states:
    print(f'[{tag}] hidden_states non disponibili')
    return
  for idx, tensor in enumerate(hidden_states):
    print(f'-livello {idx}: {tuple(tensor.shape)}')

In [ ]:
dataset = PennFudanSegmentation(DATA_ROOT)
val_len = max(1, int(len(dataset) * VAL_SPLIT))
train_len = len(dataset) - val_len
train_set, val_set = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_segformer(num_classes = 2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# --- Training ---
model.train()
train_loss = 0.0
for batch_idx, (images, masks) in enumerate(train_loader):
  images, masks = images.to(device), masks.to(device)
  optimizer.zero_grad()
  outputs = model(images)
  res = F.interpolate(outputs.logits, size=masks.shape[-2:], model='bilinear', align_corners=False)
  loss = criterion(res, masks)

  loss.backward()
  optimizer.step()
  train_loss += loss.item() * images.size(0)
train_loss /= len(train_loader.dataset)
print(f'[SegFormer] Loss training: {train_loss:.4f}')

# --- Validation ---
model.eval()
val_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
 for batch_idx, (images, masks) in enumerate(train_loader):
    images, masks = images.to(device), masks.to(device)
    outputs = model(images)
    res = F.interpolate(outputs.logits, size=masks.shape[-2:], model='bilinear', align_corners=False)
    loss = criterion(res, masks)
    val_loss += loss.item() * images.size(0)

    preds = logits.argmax(dim=1)
    correct += (preds == masks).sum().item()
    total += masks.numel()

val_loss /= len(val_loader.dataset)
pixel_acc = correct / total if total else 0.0
print(f'[SegFormer] Loss validation: {val_loss:.4f}, Pixel accuracy: {pixel_acc:.3f}')

torch.save(model.state_dict(), 'segfomer_pennfudan.pth')
print('Pesi salvati in segformer_pennfudan.pth')

## 3: Pipeline di training di MaskFormer sul dataset PennFudanPed
- Configurazioni iniziali (percorso del dataset, dim immagini, checkpoint Hugging Face)
- Creazione del dataset personalizzato
- Collate function personalizzata (return liste di immagini e maschere)
- Costruzione del modello Maskormer (MaskFormerImageProcessor e MaskFormerForInstanceSegmentation)
- DataLoader con collate personalizzata
- Preparazione del training (divisione del dataset, DataLoader, initialization)
- Addestramento, Validazione, Salvataggio del modello (esporto i pesi)

Esercizio: Utilizzare Mask2Former di Hugging Face

In [ ]:
import numpy as np
from transformers import MaskFormerForInstanceSegmentation, MaskFormerImageProcessor

from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

ROOT_DIR = Path(__file__).resolve().parents[1]
DATA_ROOT = ROOT_DIR / 'data' / 'PennFudanPed'
IMAGE_SIZE = 256
BATCH_SIZE = 2
LEARNING_RATE = 5e-4
VAL_SPLIT = 0.2
MASKFORMER_CHECKPOINT = 'facebook/maskformer-swin-base-ade'
MASK2FORMER_CHECKPOINT = 'facebook/mask2former-swin-small-coco-panoptic'

torch.manual_seed(0)

In [ ]:
def collate_fn(batch: Sequence[Tuple[Image.Image, Image.Image]]) -> Tuple[List[Image.Image], List[Image.Image]]:
  '''
  collate_fn e' la funzione che il DataLoader usa per combinare i singoli campioni in un batch. Per default prova a impilarli in tensori (o li mette in liste/dict se gia' strutturati),
  ma quando gli elementi hanno forme variabili o tipi non direttamente stackabili (PIL, dizionari complessi) ti serve una collate personalizzata. Con le reti transformer per segmentazione
  e' piu' comodo tenere immagini e maschere come liste di PIL e farle processare dal relativo ImageProcessor, quindi collate_fn diventa il punto in cui scegli come aggregare i campioni prima
  che il processor faccia padding, normalization, ecc. In generale: collate_fn definisce la logica di impacchettamento del batch; la personalizzi quando la logica standard non e' sufficiente.
  -- Restituiamo due liste separate di immagini e maschere (in formato PIL) invece di impilarle in tensori. I processor di MaskFormer lavorano proprio con liste di oggetti eterogenei e applicano
  loro il padding/resize corretto, per cui questa collate_fn fa da ponte tra DataLoader e image processor --
  '''
  images, masks = zip(*batch)
  return list(images), list(masks)

def build_maskformer(num_classes: int) -> Tuple[MaskFormerForInstanceSegmentation, MaskFormerImageProcessor]:
  '''
  Il processor gestisce normalizzazione, resizing coerente e crea le label tensoriali (class_labels, mask_labels) nel formato atteso da MaskFormer.
  '''
  processor = MaskFormerImageProcessor.from_pretrained(
      MASKFORMER_CHECKPOINT,
      ignore_index = 255,
      raduce_labels = False,
  )
  model = MaskFormerForInstanceSegmentation.from_pretrained(
      MASKFORMER_CHECKPOINT,
      ignored_mismatched_sizes = True,
      num_labels = num_classes
  )

  # model = Mask2FormerForUniversalSegmentation.from_pretrained(MASK2FORMER_CHECKPOINT, ignore_mismatched_sizes=True, num_labels=2)

  model.config.output_hidden_state = True
  return model, processor

In [ ]:
dataset = PennFudanSegmentation(DATA_ROOT)
val_len = max(1, int(len(dataset) * VAL_SPLIT))
train_len = len(dataset) - val_len
train_set, val_set = random_split(dataset, [train_len, val_len])

# Collate personalizzata per ottenere liste di PIL e lasciar gestire padding/normalizzazione al processor di MaskFormer.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, collate_fn=collate_fn)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, processor = build_maskformer(num_classes = 2)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# --- Training ---
model.train()
train_loss = 0.0
for batch_idx, (images, masks) in enumerate(train_loader):
  # Il processor restituisce un BatchFeature (dict con pixel_values, pixel_mask, label tensors) gia' pronto per .to(device)
  batch_inputs = processor(images=list(images), segmentation_maps=list(masks), return_tensors='pt')
  # BatchFeature supporta .to(device); qui mantengo l'espansione esplicita per chiarezza ma potevo fare batch_inputs = batch_inputs.to(device)
  batch_inputs = {k: v.to(device) for k, v in batch_inputs.item()}
  optimizer.zero_grad()
  # Passiamo i tensori con **batch_inputs cosi' da espandere il dict nei parametri keyword attesi dal modello (pixel_values=..., pixel_mask=..., class_labels=..., ecc.).
  outputs = model(**batch_inputs)
  # MaskFormer calcola la loss internamente combinando le componenti (Hungarian matching, BCE, dice, ecc.)
  loss = outputs.loss
  loss.backward()
  optimizer.step()
  train_loss += loss.item() * images.size(0)
train_loss /= len(train_loader.dataset)
print(f'[MaskFormer] Loss training: {train_loss:.4f}')

# --- Validation ---
model.eval()
val_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
 for batch_idx, (images, masks) in enumerate(train_loader):
    batch_inputs = processor(images=list(images), segmentation_maps=list(masks), return_tensors='pt')
    batch_inputs = {k: v.to(device) for k, v in batch_inputs.item()}

    outputs = model(**batch_inputs)
    loss = outputs.loss
    val_loss += loss.item() * len(images)

    target_sizes = [mask.size[::-1] for mask in masks]  # (H, W)
    predictions = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)

    for pred_map, mask in zip(predictions, masks):
      gt = torch.from_numpy(np.array(mask, dtype=np.int64))
      pred_resized = pred_map.to(gt.device)
      correct += (pred_resized == gt).sum().item()
      total += gt.numel()

val_loss /= len(val_loader.dataset)
pixel_acc = correct / total if total else 0.0
print(f'[MaskFormer] Loss validation: {val_loss:.4f}, Pixel accuracy: {pixel_acc:.3f}')

torch.save(model.state_dict(), 'maskfomer_pennfudan.pth')
print('Pesi salvati in maskformer_pennfudan.pth')